# Full SOC Security Operations Copilot — End to End in One Notebook

Self-contained: every step defined here in order, no imports from `src/`.

**Pipeline:** `data → fusion → calibration → RAG → summarizer → governance-gated agent → human approval`

Sections 0–20, grouped under six banners (Generate data, Fusion, RAG, Summarizer, Governance & Agent, Run).

## 0. Setup

Find the repo root (walk up until we see `project_07_final_synthesis/`), chdir
there, and pin `SEED = 42` so every generator is reproducible.

In [1]:
from __future__ import annotations
import os, sys, json, re, csv, time, hashlib, sqlite3, inspect, argparse
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field
from typing import Any, Callable, Literal, TypedDict

import numpy as np
import pandas as pd
import requests, yaml

# Walk up from the kernel cwd to the folder containing project_07_final_synthesis/.
NB_DIR = Path.cwd()
REPO_ROOT = next(p for p in [NB_DIR.resolve(), *NB_DIR.resolve().parents]
                 if (p / "project_07_final_synthesis").is_dir())
PROJECT_DIR = REPO_ROOT / "project_07_final_synthesis"
os.chdir(REPO_ROOT)
print("cwd =", Path.cwd())

SEED = 42  # single source of determinism

# cwd-relative data locations after the chdir above.
REF_DIR          = Path("data/reference")
SYNTHETIC_DIR     = PROJECT_DIR / "data" / "synthetic"
KB_DIR            = PROJECT_DIR / "data" / "knowledge_base"
KB_JSONL          = KB_DIR / "knowledge_base.jsonl"
VECTOR_STORE_DIR  = KB_DIR / "vector_store"
AGENT_DATA_DIR    = PROJECT_DIR / "data"
for d in (REF_DIR, SYNTHETIC_DIR, KB_DIR, AGENT_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

cwd = D:\AI_Master\Udacity\capstone_projects


## 1. Schema

Column lists for the three operational tables. ID prefixes are fixed
(`SITE-001`, `EVT-000245`, `INC-000041`, `KB-00012`).

In [2]:
# Surveillance events: camera detections / anomalies.
SURVEILLANCE_COLS = ["event_id","site_id","zone_id","device_id","event_timestamp",
                     "event_type","confidence_score","anomaly","description"]
# Access logs: badge-reader events at doors.
ACCESS_COLS = ["log_id","site_id","zone_id","device_id","log_timestamp",
               "user_id","access_result","reason"]
# Incidents: fusion output. summary/action/citations filled later by the LLM step.
INCIDENT_COLS = ["incident_id","site_id","zone_id","incident_start","incident_end",
                 "incident_type","linked_event_ids","linked_log_ids",
                 "risk_score","risk_band","summary_text","recommended_action",
                 "citation_doc_ids","human_review_required","created_at"]

def empty(df_cols): return pd.DataFrame({c: pd.Series(dtype="object") for c in df_cols})

def write_parquet(df, name, out_dir=SYNTHETIC_DIR):
    # Write parquet + a CSV mirror (CSV is for reviewers/Excel).
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / f"{name}.parquet", index=False)
    df.to_csv(out_dir / f"{name}.csv", index=False, encoding="utf-8")
    return out_dir / f"{name}.parquet"

def read_parquet(name, in_dir=SYNTHETIC_DIR):
    p = Path(in_dir) / f"{name}.parquet"
    if not p.exists(): raise FileNotFoundError(f"missing: {p}. Run its generator first.")
    return pd.read_parquet(p)

# 📦 Generate data
Reference data, surveillance events, access logs.

## 2. Reference data — sites / zones / devices / users

Scaled layout: 3 sites × 4 zones (`SITE-NNN::ZONE-{A,B,C,D}`), where **ZONE-D
is restricted** at every site. ~120 devices, ~240 users. Deterministic from `SEED`.

In [3]:
N_SITES, N_ZONES = 3, 4
LETTERS = ["A","B","C","D"]; RESTRICTED = "D"

def _site(i):  return f"SITE-{i+1:03d}"
def _zone(s,z): return f"{_site(s)}::ZONE-{LETTERS[z]}"   # restricted when letter == D
def _dev(i):   return f"DEV-{i+1:03d}"
def _user(i):  return f"USR-{i+1:03d}"

def write_reference():
    # Write sites/zones/devices/users CSVs from SEED.
    sites = [{"site_id": _site(i), "site_name": f"Building-{i+1}", "timezone": "UTC"}
             for i in range(N_SITES)]
    zones, devices = [], []
    counter = 0
    for s in range(N_SITES):
        for z in range(N_ZONES):
            zones.append({"zone_id": _zone(s,z), "site_id": _site(s),
                          "zone_name": f"Zone-{z+1}", "restricted": LETTERS[z] == RESTRICTED})
            # 7 cameras + 2 badge readers + 1 door per zone.
            for dtype, n in (["camera",7], ["badge_reader",2], ["door",1]):
                for _ in range(n):
                    devices.append({"device_id": _dev(counter), "site_id": _site(s),
                                    "zone_id": _zone(s,z), "device_type": dtype}); counter += 1
    # 60% employee, 20% contractor, 15% cleaner, 5% security.
    n_users = 20 * N_SITES * N_ZONES
    roles = (["employee"]*int(n_users*.6) + ["contractor"]*int(n_users*.2)
             + ["cleaner"]*int(n_users*.15) + ["security"]*int(n_users*.05))
    users = [{"user_id": _user(i), "site_id": _site(i % N_SITES), "role": roles[i],
              "authorized_zones": _zone(i % N_SITES, i % N_ZONES)} for i in range(n_users)]
    for name, rows in [("sites",sites),("zones",zones),("devices",devices),("users",users)]:
        with (REF_DIR / f"{name}.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return sites, zones, devices, users

sites, zones, devices, users = (pd.read_csv(REF_DIR/f"{n}.csv")
                                for n in ("sites","zones","devices","users"))
if not sites.shape[0]:
    sites, zones, devices, users = write_reference()
    sites, zones, devices, users = (pd.read_csv(REF_DIR/f"{n}.csv")
                                    for n in ("sites","zones","devices","users"))
restricted = zones.loc[zones["restricted"], "zone_id"].tolist()
print(f"sites={len(sites)} zones={len(zones)} devices={len(devices)} users={len(users)}")
print("restricted:", restricted)

sites=3 zones=12 devices=120 users=240
restricted: ['SITE-001::ZONE-D', 'SITE-002::ZONE-D', 'SITE-003::ZONE-D']


## 3. Surveillance events

50 camera events. **Exactly 3 anomalies** (fixed count, not a per-row coin flip),
placed in the restricted zone with high confidence (Beta(8,2), mean ~0.8) so the
intrusion rule fires naturally.

In [4]:
N_EVENTS, N_ANOMALIES, SIM_HOURS = 50, 3, 24
EVENT_TYPES = ["person_detected","person_detected","person_detected","vehicle_detected","anomaly"]

def generate_events():
    rng = np.random.default_rng(SEED)
    cams = devices[devices["device_type"] == "camera"].reset_index(drop=True)
    base = datetime(2026,7,1, tzinfo=timezone.utc)
    anomaly_idx = set(rng.choice(N_EVENTS, N_ANOMALIES, replace=False))  # exactly 3
    rows = []
    for i in range(N_EVENTS):
        cam = cams.iloc[i % len(cams)]
        is_anom = i in anomaly_idx
        zone = restricted[0] if is_anom else cam["zone_id"]          # anomalies -> restricted
        ts = base + timedelta(seconds=int(rng.uniform(0, SIM_HOURS*3600)))
        conf = float(np.clip(rng.beta(8,2), 0, 1))                   # skews high
        etype = "anomaly" if is_anom else str(rng.choice(EVENT_TYPES))
        desc = f"Camera {cam['device_id']} {'flagged an anomaly' if is_anom else 'detected a '+etype.replace('_',' ')} in {zone}."
        rows.append({"event_id": f"EVT-{i+1:06d}", "site_id": cam["site_id"], "zone_id": zone,
                      "device_id": cam["device_id"], "event_timestamp": ts, "event_type": etype,
                      "confidence_score": round(conf,3), "anomaly": is_anom, "description": desc})
    df = pd.DataFrame(rows)
    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
    return df[SURVEILLANCE_COLS]

events = generate_events()
write_parquet(events, "surveillance_events")
an = int(events["anomaly"].sum())
print(f"rows={len(events)} anomalies={an} ({an/len(events):.1%})")

rows=50 anomalies=3 (6.0%)


## 4. Access logs

200 badge-reader events, ~87% granted / 8% denied / 5% invalid. Two deterministic
injections so the rules fire every run: a **4-denial burst** in one zone (trips
"≥3 denials in an hour"), and **one tailgate** at a marked door (trips the
tailgate rule). Tailgate is never sampled — only injected.

In [5]:
N_LOGS, SIM_HOURS = 200, 24
BURST_USER, BURST_COUNT = "USR-005", 4
TAILGATE_DEV = "DEV-008"

def _result(rng):
    r = rng.random()
    if r < .87: return "granted", "ok"
    if r < .95: return "denied", str(rng.choice(["expired","wrong_zone","revoked"]))
    return "invalid", "unknown_badge"

def generate_logs():
    rng = np.random.default_rng(SEED)
    readers = devices[devices["device_type"] == "badge_reader"].reset_index(drop=True)
    base = datetime(2026,7,1, tzinfo=timezone.utc)
    uids = users["user_id"].tolist()
    rows = []
    for i in range(N_LOGS):
        reader = readers.iloc[i % len(readers)]
        result, reason = _result(rng)
        uid = BURST_USER if i < BURST_COUNT else str(rng.choice(uids))   # first 4 = burst user
        ts = base + timedelta(seconds=int(rng.uniform(0, SIM_HOURS*3600)))
        rows.append({"log_id": f"LOG-{i+1:06d}", "site_id": reader["site_id"], "zone_id": reader["zone_id"],
                     "device_id": reader["device_id"], "log_timestamp": ts, "user_id": uid,
                     "access_result": result, "reason": reason})
    df = pd.DataFrame(rows)
    # Override the burst rows: 4 denials clustered in one zone within an hour.
    start = base + timedelta(hours=10)
    for k in range(BURST_COUNT):
        df.at[k,"log_timestamp"] = start + timedelta(minutes=k*5)
        df.at[k,"access_result"] = "denied"; df.at[k,"reason"] = "revoked"
        df.at[k,"device_id"] = readers.iloc[0]["device_id"]; df.at[k,"zone_id"] = readers.iloc[0]["zone_id"]
    # One deterministic tailgate at the marked door, right after the burst.
    df.at[BURST_COUNT+2,"device_id"] = TAILGATE_DEV
    df.at[BURST_COUNT+2,"access_result"] = "tailgate"; df.at[BURST_COUNT+2,"reason"] = "forced_door"
    df.at[BURST_COUNT+2,"log_timestamp"] = start + timedelta(minutes=BURST_COUNT*5+1)
    df = df.sort_values("log_timestamp").reset_index(drop=True)
    df["log_id"] = [f"LOG-{i+1:06d}" for i in range(len(df))]
    df["log_timestamp"] = pd.to_datetime(df["log_timestamp"], utc=True)
    return df[ACCESS_COLS]

logs = generate_logs()
write_parquet(logs, "access_logs")
print(f"rows={len(logs)} denied={int((logs['access_result']=='denied').sum())} "
      f"tailgate={int((logs['access_result']=='tailgate').sum())}")

rows=200 denied=18 tailgate=1


# 🧩 Fusion
Rule detectors, risk scoring, incident materialization.

## 5. Fusion rules — four detectors

Rules-first (P3 lesson: report what each rule fired on, not a black-box score).
Each `detect_*` returns candidate dicts; the scorer turns them into `INC-` rows.

| rule | fires when |
|---|---|
| `intrusion_restricted` | anomaly in a restricted zone with confidence ≥ **0.85** |
| `repeated_denials` | ≥ **3** denials in one zone within **60 min** |
| `cross_anomaly` | anomaly + unusual access in the same zone within **10 min** |
| `tailgate_door` | a tailgate log followed by `forced_door` within **10 min** |

`0.85` is a *confidence* trigger (detection); the `80` *risk* gate comes later.

In [6]:
# Rule thresholds — single source of truth for the rules.
CONF_MIN, DENY_MIN, DENY_WIN = 0.85, 3, 60
CROSS_WIN, TAILGATE_WIN = 10, 10
RULE_NAMES = {"intrusion_restricted":"Restricted-Zone Intrusion",
              "repeated_denials":"Repeated Badge Denials",
              "cross_anomaly":"Surveillance + Access Anomaly Correlation",
              "tailgate_door":"Tailgating + Door Activity"}

def _intrusion(events, zones):
    restricted = set(zones.loc[zones["restricted"], "zone_id"])
    ev = events[events["anomaly"] & events["zone_id"].isin(restricted)
                & (events["confidence_score"] >= CONF_MIN)]
    return [{"incident_type":"suspected_unauthorized_entry", "rule":"intrusion_restricted",
             "linked_event_ids":[r.event_id], "linked_log_ids":[],
             "incident_start":r.event_timestamp, "incident_end":r.event_timestamp,
             "site_id":r.site_id, "zone_id":r.zone_id} for r in ev.itertuples(index=False)]

def _denials(logs):
    win = timedelta(minutes=DENY_WIN)
    out = []
    for zone, g in logs[logs["access_result"]=="denied"].sort_values("log_timestamp").groupby("zone_id"):
        rows = list(g.itertuples(index=False))
        for i in range(len(rows)):
            w = [rows[j] for j in range(i, len(rows))
                 if (rows[j].log_timestamp - rows[i].log_timestamp) <= win]
            if len(w) >= DENY_MIN:
                out.append({"incident_type":"repeated_badge_denials","rule":"repeated_denials",
                            "linked_event_ids":[], "linked_log_ids":[r.log_id for r in w],
                            "incident_start":min(r.log_timestamp for r in w),
                            "incident_end":max(r.log_timestamp for r in w),
                            "site_id":rows[0].site_id, "zone_id":zone}); break
    return out

def _cross(events, logs):
    win = timedelta(minutes=CROSS_WIN)
    unusual = logs[logs["access_result"].isin(["denied","invalid","tailgate"])]
    out = []
    for ev in events[events["anomaly"]].itertuples(index=False):
        c = unusual[(unusual["zone_id"]==ev.zone_id)
                    & unusual["log_timestamp"].between(ev.event_timestamp-win, ev.event_timestamp+win)]
        if not c.empty:
            c = c.iloc[0]
            out.append({"incident_type":"cross_anomaly_correlation","rule":"cross_anomaly",
                        "linked_event_ids":[ev.event_id], "linked_log_ids":[c["log_id"]],
                        "incident_start":min(ev.event_timestamp,c["log_timestamp"]),
                        "incident_end":max(ev.event_timestamp,c["log_timestamp"]),
                        "site_id":ev.site_id, "zone_id":ev.zone_id})
    return out

def _tailgate(logs):
    win = timedelta(minutes=TAILGATE_WIN)
    tails = logs[logs["access_result"]=="tailgate"]
    doors = logs[logs["reason"]=="forced_door"]
    out = []
    for t in tails.itertuples(index=False):
        n = doors[(doors["zone_id"]==t.zone_id)
                  & doors["log_timestamp"].between(t.log_timestamp, t.log_timestamp+win)]
        if not n.empty:
            nb = n.iloc[0]
            linked = list(dict.fromkeys([t.log_id, nb["log_id"]]))   # dedupe (the injected tailgate is both)
            out.append({"incident_type":"tailgate_door_activity","rule":"tailgate_door",
                        "linked_event_ids":[], "linked_log_ids":linked,
                        "incident_start":min(t.log_timestamp,nb["log_timestamp"]),
                        "incident_end":max(t.log_timestamp,nb["log_timestamp"]),
                        "site_id":t.site_id, "zone_id":t.zone_id})
    return out

def all_candidates(events, logs, zones, devices):
    return _intrusion(events, zones) + _denials(logs) + _cross(events, logs) + _tailgate(logs)

from collections import Counter
raw = all_candidates(events, logs, zones, devices)
print("raw candidates:", len(raw), dict(Counter(c["rule"] for c in raw)))

raw candidates: 3 {'intrusion_restricted': 1, 'repeated_denials': 1, 'tailgate_door': 1}


## 6. Risk scorer

Per-rule **base** + **size bonus** (more linked evidence) + **confidence bonus**
(surveillance rules), capped at 100.

```
critical >= 80   (forces human review — the 80 gate)
high     >= 60
medium   >= 40
low      <  40
```

In [7]:
# Per-rule base risk; +3 per extra linked signal (cap 15); +confidence (cap 10).
BASE = {"intrusion_restricted":70, "repeated_denials":55, "cross_anomaly":60, "tailgate_door":65}
LINK_PER, LINK_CAP = 3, 15
CONF_CAP = 10

def band(score):
    if score >= 80: return "critical"
    if score >= 60: return "high"
    if score >= 40: return "medium"
    return "low"

def score_candidate(c, conf_by_event=None):
    base = BASE.get(c["rule"], 40)
    n_links = len(c.get("linked_event_ids",[])) + len(c.get("linked_log_ids",[]))
    size_bonus = min(n_links * LINK_PER, LINK_CAP)
    conf_bonus = 0.0
    if conf_by_event:
        confs = [conf_by_event[e] for e in c.get("linked_event_ids",[]) if e in conf_by_event]
        if confs: conf_bonus = min((sum(confs)/len(confs)) * CONF_CAP, CONF_CAP)
    score = min(int(base + size_bonus + conf_bonus), 100)
    return {**c, "risk_score": score, "risk_band": band(score)}

# demo on the raw candidates
conf_by_event = dict(zip(events["event_id"], events["confidence_score"].astype(float)))
for c in raw[:6]:
    s = score_candidate(c, conf_by_event)
    print(f"{s['rule']:22s} score={s['risk_score']:3d} band={s['risk_band']}")

intrusion_restricted   score= 82 band=critical
repeated_denials       score= 67 band=high
tailgate_door          score= 68 band=high


## 7. Incidents — dedup + score + materialize

Dedup on `(rule, sorted linked ids)`, score each survivor, materialize to
`INCIDENT_COLS`, write `incidents.parquet`. `summary` / `action` / `citations`
are left empty — the RAG+LLM step fills them later.

In [8]:
def _key(c):
    return (c["rule"], tuple(sorted(c.get("linked_event_ids",[]))),
            tuple(sorted(c.get("linked_log_ids",[]))))

def build_incidents(events, logs, zones, devices):
    # dedup
    seen, deduped = set(), []
    for c in all_candidates(events, logs, zones, devices):
        k = _key(c)
        if k not in seen: seen.add(k); deduped.append(c)
    # score
    conf_by_event = dict(zip(events["event_id"], events["confidence_score"].astype(float)))
    scored = [score_candidate(c, conf_by_event) for c in deduped]
    # materialize
    now = datetime.now(timezone.utc)
    rows = [{"incident_id": f"INC-{i+1:06d}", "site_id":c["site_id"], "zone_id":c["zone_id"],
             "incident_start":c["incident_start"], "incident_end":c["incident_end"],
             "incident_type":c["incident_type"],
             "linked_event_ids":",".join(c.get("linked_event_ids",[])),
             "linked_log_ids":",".join(c.get("linked_log_ids",[])),
             "risk_score":int(c["risk_score"]), "risk_band":c["risk_band"],
             "summary_text":"", "recommended_action":"", "citation_doc_ids":"",
             "human_review_required": c["risk_band"]=="critical", "created_at":now,
             "_rule":c["rule"]} for i,c in enumerate(scored)]
    df = pd.concat([empty(INCIDENT_COLS), pd.DataFrame(rows)], ignore_index=True)
    df["risk_score"] = df["risk_score"].astype(int)
    df["human_review_required"] = df["human_review_required"].astype(bool)
    for col in ("incident_start","incident_end","created_at"):
        df[col] = pd.to_datetime(df[col], utc=True)
    return df

incidents = build_incidents(events, logs, zones, devices)
write_parquet(incidents, "incidents")
print(f"rows={len(incidents)} by_band={incidents['risk_band'].value_counts().to_dict()} "
      f"critical={int(incidents['human_review_required'].sum())}")
incidents[["incident_id","incident_type","risk_score","risk_band","_rule","human_review_required"]]

rows=3 by_band={'high': 2, 'critical': 1} critical=1


,incident_id,incident_type,risk_score,risk_band,_rule,human_review_required
0,INC-000001,suspected_unauthorized_entry,82,critical,intrusion_restricted,True
1,INC-000002,repeated_badge_denials,67,high,repeated_denials,False
2,INC-000003,tailgate_door_activity,68,high,tailgate_door,False


### 2b. P2 → P7 threshold calibration

P2 warned the 0.85 threshold should be *validated against precision/recall
before deployment.* This re-runs P2's two tests on the slice just fused:
- **H1 chi-square** — zone restrictiveness × unusual access → negligible predictor (premise holds).
- **H2 Welch t-test** — anomaly vs normal confidence → effect stronger here; **recall@0.85 is the hard finding**.

Mirrors `tests/test_threshold_calibration.py`; shown inline so the reader sees the numbers next to fusion.

In [9]:
from scipy.stats import chi2_contingency, ttest_ind

# P2's headline numbers (calibration target).
P2_P, P2_V, P2_D, P2_MEAN = 0.0341, 0.0221, 0.21, 0.825
THRESH = 0.85

def _restricted(z):  # ZONE-D is restricted.
    return isinstance(z, str) and z.rsplit("ZONE-",1)[-1].strip() == "D"

# H1: zone restrictiveness x unusual access outcome.
acc = logs.copy()
acc["zone_restricted"] = acc["zone_id"].map(_restricted)
acc["unusual"] = acc["access_result"].isin(["denied","invalid","tailgate"])
ct = pd.crosstab(acc["zone_restricted"], acc["unusual"])
chi2, p, _, _ = chi2_contingency(ct)
n = int(ct.values.sum()); v = float(np.sqrt(chi2 / (n * (min(ct.shape)-1))))

# H2: Welch t-test, anomaly vs normal confidence.
anom = events.loc[events["anomaly"], "confidence_score"].astype(float).values
norm = events.loc[~events["anomaly"], "confidence_score"].astype(float).values
t, p_t = ttest_ind(anom, norm, equal_var=False)
pooled = float(np.sqrt((anom.var(ddof=1) + norm.var(ddof=1)) / 2))
d = float((anom.mean() - norm.mean()) / pooled)
recall = float((anom >= THRESH).mean())

print(f"P2 -> P7 calibration (n_events={len(events)}, n_access={len(logs)})\n")
print(f"H1 chi-square  P7: p={p:.4g} V={v:.4f}   | P2: p={P2_P} V={P2_V}")
print(f"  -> zone restrictiveness stays *negligible* (fusion premise holds).\n")
print(f"H2 t-test      P7: d={d:.3f} mean_anom={anom.mean():.3f} recall@0.85={recall:.0%}  | P2: d={P2_D} mean={P2_MEAN}")
print(f"  -> effect *stronger* here; 0.85 catches only {recall:.0%} of anomalies -- HARD FINDING.\n")
pd.DataFrame([
 {"test":"H1 chi-square (zone x outcome)","P2_p":P2_P,"P7_p":p,"P2_effect":"V=0.022","P7_effect":f"V={v:.3f}","verdict":"negligible (premise holds)"},
 {"test":"H2 t-test (anomaly vs normal)","P2_p":1.36e-14,"P7_p":p_t,"P2_effect":"d=0.21","P7_effect":f"d={d:.2f}","verdict":f"stronger; recall@0.85={recall:.0%} is the hard finding"},
])

P2 -> P7 calibration (n_events=50, n_access=200)

H1 chi-square  P7: p=0.5503 V=0.0422   | P2: p=0.0341 V=0.0221
  -> zone restrictiveness stays *negligible* (fusion premise holds).

H2 t-test      P7: d=0.599 mean_anom=0.850 recall@0.85=33%  | P2: d=0.21 mean=0.825
  -> effect *stronger* here; 0.85 catches only 33% of anomalies -- HARD FINDING.



,test,P2_p,P7_p,P2_effect,P7_effect,verdict
0,H1 chi-square (zone x outcome),3.410000e-02,0.550282,V=0.022,V=0.042,negligible (premise holds)
1,H2 t-test (anomaly vs normal),1.360000e-14,0.325032,d=0.21,d=0.60,stronger; recall@0.85=33% is the hard finding


# 📚 RAG
Policy KB, Chroma vector store, MMR + category routing.

## 8. Knowledge base — 5 policy docs (inlined)

Inlined so the notebook is self-contained. The **category** is the 1:1 map from
the fusion layer's `incident_type` to the KB — that's what category routing uses.

In [10]:
KB_DOCS = [
  {"doc_id":"KB-00001","title":"Restricted-Zone Intrusion Response","category":"intrusion",
   "body":"When a surveillance anomaly with confidence >= 0.85 occurs in a restricted zone, treat it as a suspected unauthorized entry. First action: dispatch on-site security to the zone within 5 minutes. Second: pull the camera feed 10 min before/after and archive it. Third: check access_logs for any denied/invalid badge in the same zone within 10 minutes; if found, escalate to critical and a human reviewer must verify before any physical response. Do not lock doors automatically - door-lock actions need human approval. Retention: 90 days minimum."},
  {"doc_id":"KB-00002","title":"Repeated Badge Denials Response","category":"denials",
   "body":"Three or more denied/invalid badge attempts in the same zone within 60 minutes is a credential-attack indicator. First: review user_id history across all sites for 7 days. Second: if the same user_id has denials in more than one site/zone, suspend the badge pending identity verification. Third: notify the duty manager. Do not auto-disable the account - disable needs human approval. Retention: 90 days. Privacy: user_id is internal; do not include the badge holder's name in any external summary."},
  {"doc_id":"KB-00003","title":"Surveillance + Access Anomaly Correlation","category":"correlation",
   "body":"When a surveillance anomaly (any confidence) and an unusual access event (denied, invalid, or tailgate) occur in the same zone within 10 minutes, treat them as one correlated incident - the combined signal is stronger than either alone. First: dispatch security if the access event is denied or tailgate. Second: pull the camera clip covering the anomaly. Third: if both are in a restricted zone, escalate to critical. Always cite this doc when recommending a correlated response."},
  {"doc_id":"KB-00004","title":"Tailgating + Door Activity Response","category":"tailgate",
   "body":"A tailgate log (one badge grant, multiple persons passing) followed by door-sensor activity (forced_door reason) within 10 minutes in the same zone indicates a possible physical breach. First: dispatch security. Second: hold the door locked pending on-site arrival; do not auto-unlock. Third: review the camera feed for the badge_id used in the tailgate. If that badge belongs to a different role (e.g. cleaner in a server room), escalate to critical. Human approval is required before any door-state change."},
  {"doc_id":"KB-00005","title":"Privacy, PII, and Access-Log Retention","category":"privacy",
   "body":"Access logs may contain user_id (internal) but must NOT include badge holder names, contact details, or biometric data in any external-facing summary. The summarizer must redact any user_name, email, or phone fields. Retention: access logs 90 days; surveillance event metadata (no raw video) 180 days; incident records 1 year. Human review is mandatory for any incident with risk_band = critical. The copilot never auto-resolves a critical incident."},
]
# Write the JSONL the loader reads (keeps on-disk format identical to the project).
with KB_JSONL.open("w", encoding="utf-8") as f:
    for d in KB_DOCS: f.write(json.dumps(d, ensure_ascii=False) + "\n")
for d in KB_DOCS: print(f"  {d['doc_id']} [{d['category']:11s}] {d['title']}")

  KB-00001 [intrusion  ] Restricted-Zone Intrusion Response
  KB-00002 [denials    ] Repeated Badge Denials Response
  KB-00003 [correlation] Surveillance + Access Anomaly Correlation
  KB-00004 [tailgate   ] Tailgating + Door Activity Response
  KB-00005 [privacy    ] Privacy, PII, and Access-Log Retention


## 9. KB loader → Chroma vector store

**Native `chromadb` + `sentence-transformers`** — no LangChain wrappers. A 5-doc
KB with one embedder has nothing to swap, so the wrapper would be an interface
with a single implementation. We embed `title + body` (title carries signal for
short docs) and store `doc_id/title/category` metadata so retrieval returns ids.

In [11]:
import chromadb
from sentence_transformers import SentenceTransformer

COLLECTION = "p7_policy_kb"
EMBED_MODEL = "all-MiniLM-L6-v2"

def build_vector_store():
    # Embed the KB docs into a persistent Chroma collection (cosine space).
    client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
    try: client.delete_collection(COLLECTION)      # idempotent rebuild
    except Exception: pass
    col = client.get_or_create_collection(COLLECTION, metadata={"hnsw:space":"cosine"})
    model = SentenceTransformer(EMBED_MODEL)
    texts = [f"{d['title']}\n{d['body']}" for d in KB_DOCS]
    col.add(ids=[d["doc_id"] for d in KB_DOCS], documents=texts,
            metadatas=[{"doc_id":d["doc_id"],"title":d["title"],"category":d["category"]} for d in KB_DOCS],
            embeddings=model.encode(texts, normalize_embeddings=True).tolist())
    return len(KB_DOCS), VECTOR_STORE_DIR

n, store_dir = build_vector_store()
print(f"built {COLLECTION}: {n} docs -> {store_dir}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

built p7_policy_kb: 5 docs -> D:\AI_Master\Udacity\capstone_projects\project_07_final_synthesis\data\knowledge_base\vector_store


## 10. Retriever — MMR with category routing

**MMR** re-ranks so each of k=3 docs is relevant **but not a repeat** of the others:
`score = λ·relevance − (1−λ)·redundancy`, λ=0.5.

**Category routing:** the fusion layer already knows the `incident_type`, so the
matching category gets a **+0.3** bonus before MMR — guaranteeing the right
policy ranks first. The returned `score` is the raw cosine (pre-bonus) for transparency.

In [12]:
TOP_K, POOL_N, LAMBDA, BONUS = 3, 10, 0.5, 0.3
# incident_type -> KB category (the 1:1 alignment between rule layer and KB).
TYPE_TO_CAT = {"suspected_unauthorized_entry":"intrusion", "repeated_badge_denials":"denials",
               "cross_anomaly_correlation":"correlation", "tailgate_door_activity":"tailgate"}
_MODEL = None  # lazy singleton: the ~3s model load happens once per process

def _cos(a, b):
    a = a/(np.linalg.norm(a)+1e-12); b = b/(np.linalg.norm(b,axis=1,keepdims=True)+1e-12)
    return b @ a

def _mmr(rel, embs, k, lam=LAMBDA):
    # Greedy MMR. O(m*k) — fine at m<=10; cap m with ANN for a larger KB.
    k = min(k, len(rel)); sel, rem, max2sel = [], list(range(len(rel))), np.full(len(rel), -np.inf)
    while len(sel) < k and rem:
        best, best_s = None, -np.inf
        for i in rem:
            div = 0.0 if not sel else max2sel[i]
            s = lam*rel[i] - (1-lam)*div
            if s > best_s: best_s, best = s, i
        sel.append(best); rem.remove(best)
        max2sel = np.maximum(max2sel, _cos(embs[best], embs).ravel())
    return sel

def retrieve(query, k=TOP_K, category_hint=None):
    # MMR retrieve. category_hint boosts matching-category docs (MMR ordering only).
    col = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR)).get_collection(COLLECTION)
    pool = col.query(query_texts=[query], n_results=min(POOL_N, col.count()))
    ids, docs, metas = pool["ids"][0], pool["documents"][0], pool["metadatas"][0]
    if not ids: return []
    embs = np.asarray(col.get(ids=ids, include=["embeddings"])["embeddings"], dtype=np.float32)
    global _MODEL
    if _MODEL is None: _MODEL = SentenceTransformer(EMBED_MODEL)
    rel = _cos(_MODEL.encode([query], normalize_embeddings=True)[0].astype(np.float32), embs)
    mmr = rel.copy()
    if category_hint:
        for i,m in enumerate(metas):
            if m.get("category") == category_hint: mmr[i] += BONUS
    order = _mmr(mmr, embs, k=k)
    return [{"doc_id":metas[i]["doc_id"],"title":metas[i]["title"],"category":metas[i]["category"],
             "body":docs[i],"score":float(rel[i])} for i in order]

def retrieve_for_incident(incident_type, query, k=TOP_K):
    # Route on the fusion-known incident_type, then MMR.
    return retrieve(query, k=k, category_hint=TYPE_TO_CAT.get(incident_type))

# demo: route on a tailgate incident and watch the right doc rank first.
res = retrieve_for_incident("tailgate_door_activity",
                            "tailgate followed by forced door activity in a restricted zone")
for r in res:
    print(f"  [{r['doc_id']}] ({r['category']:11s}) score={r['score']:.3f}  {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  [KB-00004] (tailgate   ) score=0.560  Tailgating + Door Activity Response
  [KB-00002] (denials    ) score=0.710  Repeated Badge Denials Response
  [KB-00003] (correlation) score=0.514  Surveillance + Access Anomaly Correlation


# ✍️ Summarizer
Groq LLM with a citation guard (stub fallback so it runs without a key).

## 11. Summarizer — Groq LLM with a citation guard

Groq `llama-3.1-8b-instant` via plain `requests` (no OpenAI SDK). The guard makes
"hallucinated policies are bugs" enforceable on a free 8B model:
1. parse `KB-XXXXX` ids from the output; 2. keep only ids in the retrieved set;
3. if none survive → re-prompt **once** with the allowed ids listed; 4. else mark `needs_review`.

No `GROQ_API_KEY` → **stub mode**: a deterministic summary citing the top retrieved doc.

In [13]:
from dotenv import load_dotenv
load_dotenv(PROJECT_DIR / ".env")

GROQ_URL = os.environ.get("GROQ_BASE_URL","https://api.groq.com/openai/v1") + "/chat/completions"
GROQ_MODEL = os.environ.get("GROQ_MODEL","llama-3.1-8b-instant")
KB_ID_RE = re.compile(r"KB-\d{5}")
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))

def _chat(messages):
    # One Groq call; retries on 429/5xx with backoff, raises on 4xx.
    h = {"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}", "Content-Type":"application/json"}
    body = {"model":GROQ_MODEL,"messages":messages,"temperature":0.2}
    for attempt in range(3):
        try: r = requests.post(GROQ_URL, headers=h, json=body, timeout=60)
        except requests.RequestException: time.sleep(2**attempt); continue
        if r.status_code == 200: return r.json()["choices"][0]["message"]["content"]
        if r.status_code in (429,500,502,503,504): time.sleep(2**attempt); continue
        raise RuntimeError(f"Groq {r.status_code}: {r.text[:300]}")
    raise RuntimeError("Groq failed after 3 retries")

def _prompt(incident, docs):
    # System rule lists the allowed doc_ids explicitly; user gives the case.
    allowed = ", ".join(d["doc_id"] for d in docs)
    policy = "\n\n".join(f"[{d['doc_id']}] {d['title']} ({d['category']})\n{d['body']}" for d in docs)
    sys_ = (f"You are a SOC copilot. Write a concise analyst-facing incident summary grounded "
            f"in the provided policy docs. Every procedural claim MUST cite a doc_id from "
            f"({allowed}). Do NOT invent ids. Output JSON: summary, recommended_action, citations.")
    usr = (f"Incident {incident['incident_id']} ({incident['incident_type']}), "
           f"risk_band={incident['risk_band']}, zone={incident['zone_id']}.\n"
           f"Linked events: {incident['linked_event_ids'] or 'none'}; "
           f"linked logs: {incident['linked_log_ids'] or 'none'}.\n\n"
           f"Relevant policies:\n{policy}\n\nWrite the JSON now.")
    return [{"role":"system","content":sys_}, {"role":"user","content":usr}]

def _cites(text):
    # Pull KB-XXXXX ids from text. Coerce non-strings: the model sometimes
    # returns summary/recommended_action as a list, which would crash re.findall.
    if not isinstance(text, str):
        text = json.dumps(text) if isinstance(text,(list,dict)) else str(text)
    return list(dict.fromkeys(KB_ID_RE.findall(text)))  # dedup, order-preserving

def _flatten_citations(c):
    # c can be a list of strings, a list of {doc_id:..} dicts, or mixed -> list[str].
    out = []
    for x in (c or []):
        if isinstance(x, str): out.append(x)
        elif isinstance(x, dict):
            v = x.get('doc_id') or next(iter(x.values()), '')
            if isinstance(v, str): out.append(v)
    return out

def _parse(text):
    # Best-effort JSON parse; fall back to treating the text as the summary.
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    return {"summary":text.strip(), "recommended_action":"", "citations":_cites(text)}

def _stub(incident, docs):
    # Deterministic fallback: cite the top retrieved doc.
    top = docs[0] if docs else None
    return {"summary": f"[stub] {incident['incident_type']} in {incident['zone_id']} "
                       f"(risk={incident['risk_band']}, score={incident['risk_score']}).",
            "recommended_action": f"[stub] Follow {top['title']} ({top['doc_id']})." if top else "[stub] No policy.",
            "citations": [top["doc_id"]] if top else []}

def summarize_incident(incident):
    # Retrieve -> call (or stub) -> validate citations -> retry once -> needs_review.
    docs = retrieve_for_incident(incident["incident_type"],
                                 f"{incident['incident_type']} in zone {incident['zone_id']}; "
                                 f"events {incident['linked_event_ids']}; logs {incident['linked_log_ids']}")
    valid = {d["doc_id"] for d in docs}

    if not HAS_KEY:  # stub path
        parsed = _stub(incident, docs)
        ok = [c for c in parsed["citations"] if c in valid]
        return {**incident, "summary_text":parsed["summary"], "recommended_action":parsed["recommended_action"],
                "citation_doc_ids":",".join(ok), "_summary_status":"stub" if ok else "needs_review"}

    # LLM path: call, validate, retry once with a stricter prompt if empty.
    parsed = _parse(_chat(_prompt(incident, docs)))
    parsed["citations"] = (_cites(parsed.get("summary","")) + _cites(parsed.get("recommended_action",""))
                         + _flatten_citations(parsed.get("citations")))
    ok = [c for c in parsed["citations"] if c in valid]
    if not ok:
        msgs = _prompt(incident, docs)
        msgs[1]["content"] += f"\n\nIMPORTANT: cite at least one of: {', '.join(sorted(valid))}."
        parsed = _parse(_chat(msgs))
        parsed["citations"] = (_cites(parsed.get("summary","")) + _cites(parsed.get("recommended_action",""))
                         + _flatten_citations(parsed.get("citations")))
        ok = [c for c in parsed["citations"] if c in valid]
    if not ok:
        _s = parsed.get("summary","")
        _s = "\n".join(map(str,_s)) if isinstance(_s,list) else str(_s)
        return {**incident, "summary_text":"[needs_review: no valid citation] "+_s[:200],
                "recommended_action":parsed.get("recommended_action",""), "citation_doc_ids":"",
                "_summary_status":"needs_review"}
    summ = parsed.get("summary","")
    summ = "\n".join(map(str,summ)) if isinstance(summ,list) else str(summ)
    act = parsed.get("recommended_action","")
    act = "\n".join(map(str,act)) if isinstance(act,list) else str(act)
    return {**incident, "summary_text":summ.strip(),
            "recommended_action":act.strip(),
            "citation_doc_ids":",".join(ok), "_summary_status":"ok"}

# demo on the first incident.
inc0 = incidents.iloc[0][["incident_id","incident_type","risk_band","risk_score","zone_id",
                          "linked_event_ids","linked_log_ids"]].to_dict()
out = summarize_incident(inc0)
print(f"mode={'LLM' if HAS_KEY else 'STUB'}  status={out['_summary_status']}")
print("summary:", out["summary_text"]); print("cites  :", out["citation_doc_ids"] or "-")

mode=LLM  status=ok
summary: {'incident_id': 'INC-000001', 'risk_band': 'critical', 'zone': 'SITE-001::ZONE-D', 'description': 'Suspected unauthorized entry in restricted zone with confidence >= 0.85. Linked events: EVT-000038. No linked logs.'}
cites  : KB-00001,KB-00001,KB-00001,KB-00001,KB-00005


# 🛡️ Governance & Agent
Policy gate, governance nodes, graph (with the plan-loop FIX), domain tools, prompts, copilot wiring.

## 12. Policy gate — the reuse linchpin

A small **predicate evaluator** (`gt/ge/lt/le/eq/ne/in/regex_match`) over a
`constraints` map. The *same mechanism* that enforced the donor's spend cap
enforces our `risk_band_score ≥ 80` human-review gate. **One mechanism, two domains.**

- `incident.escalate`: side-effect, `require_human`, constraint `risk_band_score >= 80`.
- `case.close`: `allow: false` — **hard block**; a human closes a case.

In [14]:
# Predicate evaluator: field -> {pred: operand}. Reused verbatim across domains.
PREDICATES = {"gt":lambda v,o: v is not None and v>o, "ge":lambda v,o: v is not None and v>=o,
              "lt":lambda v,o: v is not None and v<o, "le":lambda v,o: v is not None and v<=o,
              "eq":lambda v,o: v==o, "ne":lambda v,o: v!=o, "in":lambda v,o: v in o,
              "regex_match":lambda v,o: bool(re.search(o,str(v))) if v is not None else False}

def eval_constraints(args, constraints):
    # Violation strings for any failed constraint. Missing field = violation (fail closed).
    out = []
    for field, preds in constraints.items():
        v = args.get(field) if args else None
        for name, operand in preds.items():
            pred = PREDICATES.get(name)
            if pred is None: out.append(f"unknown_predicate:{name}")
            elif not pred(v, operand): out.append(f"constraint_failed:{field}:{name}:{operand}")
    return out

@dataclass
class ActionRule:
    name: str; side_effect: bool; allow: bool; require_human: bool = False
    constraints: dict = field(default_factory=dict); require_fields: list = field(default_factory=list)
    block_reason: str = ""

@dataclass
class ReviewDecision:
    allow: bool; require_human: bool; violations: list = field(default_factory=list); reason: str = ""
    @property
    def route(self):  # where the graph goes next
        return "block" if not self.allow else ("require_human" if self.require_human else "allow")

@dataclass
class Policy:
    domain: str; actions: dict; redaction_patterns: list = field(default_factory=list)
    redaction_enabled: bool = True

    @classmethod
    def from_dict(cls, data):
        acts = {a["name"]: ActionRule(a["name"], bool(a.get("side_effect",False)),
                 bool(a.get("allow",False)), bool(a.get("require_human",False)),
                 a.get("constraints",{}) or {}, list(a.get("require_fields",[]) or []),
                 a.get("block_reason","") or "") for a in data.get("actions",[])}
        pii = data.get("pii_redaction",{}) or {}
        pats = [RedactionPattern(p["name"],p["regex"],p["replacement"]) for p in pii.get("patterns",[])]
        return cls(data.get("domain","unspecified"), acts, pats, bool(pii.get("enabled",True)))

    def evaluate(self, action, args):
        # Pure, no LLM/tools — the reviewer cannot be talked into approving.
        rule = self.actions.get(action)
        if rule is None: return ReviewDecision(False, False, ["unknown_action"], f"{action} not in policy.")
        if not rule.allow: return ReviewDecision(False, rule.require_human, ["action_not_allowed"],
                                                  rule.block_reason or f"{action} not permitted.")
        violations = [f"missing_required_field:{rf}" for rf in rule.require_fields if not args or args.get(rf) in (None,"")]
        violations += eval_constraints(args or {}, rule.constraints)
        if violations: return ReviewDecision(False, rule.require_human or rule.side_effect, violations,
                                             f"{action} failed policy: {violations}")
        return ReviewDecision(True, rule.require_human, [], f"{action} permitted by policy.")

@dataclass
class RedactionPattern: name: str; regex: str; replacement: str

# The SOC policy (same schema as the donor's property-management policy).
SOC_POLICY = {"domain":"security_operations",
  "pii_redaction":{"enabled":True,"patterns":[
    {"name":"email","regex":r"[\w.+-]+@[\w-]+\.[\w.-]+","replacement":"[EMAIL-REDACTED]"},
    {"name":"phone","regex":r"\b\d{10}\b","replacement":"[PHONE-REDACTED]"},
    {"name":"ssn","regex":r"\b\d{3}-\d{2}-\d{4}\b","replacement":"[SSN-REDACTED]"}]},
  "actions":[
    {"name":"incident.fuse","side_effect":False,"allow":True},
    {"name":"incident.score","side_effect":False,"allow":True},
    {"name":"sop.retrieve","side_effect":False,"allow":True},
    {"name":"incident.summarize","side_effect":False,"allow":True},
    {"name":"incident.escalate","side_effect":True,"allow":True,"require_human":True,
     "constraints":{"risk_band_score":{"ge":80}},"require_fields":["incident_id","risk_band_score"],
     "block_reason":"Escalation requires a critical-band incident and human approval."},
    {"name":"case.close","side_effect":True,"allow":False,"require_human":True,
     "block_reason":"Case closure is outside agent authority; a human analyst must close it."}]}
policy = Policy.from_dict(SOC_POLICY)

# self-check: the two-domain linchpin.
print("escalate@85 ->", policy.evaluate("incident.escalate",{"incident_id":"INC-1","risk_band_score":85}).route)
print("escalate@70 ->", policy.evaluate("incident.escalate",{"incident_id":"INC-1","risk_band_score":70}).route)
print("case.close  ->", policy.evaluate("case.close",{"incident_id":"INC-1"}).route)

escalate@85 -> require_human
escalate@70 -> block
case.close  -> block


## 13. Governance state — dataclasses flowing through every node

Domain-agnostic. `domain_state` is an opaque dict the domain fills (e.g.
`incident_id`); governance nodes never read inside it.

In [15]:
@dataclass
class PlanStep:    action: str; reason: str=""; expected_side_effect: bool=False; args: dict=field(default_factory=dict)
@dataclass
class ActionIntent: action: str; args: dict=field(default_factory=dict); side_effect: bool=False; cost_estimate: float|None=None
@dataclass
class ToolResult:   tool: str; ok: bool; summary: str; payload: Any=None
@dataclass
class GovReview(ReviewDecision): pass  # graph-flavored alias; ReviewDecision carries route

class AgentState(TypedDict, total=False):
    user_id: str; turn_id: str; messages: list; redacted_text: str
    plan: list; step_index: int; current_action: Any; review: Any; tool_result: Any
    domain_state: dict; iteration: int; status: str

## 14. Audit log + memory + PII redactor

**Audit log:** hash-chained JSONL — each line stores the hash of the previous
line, so tampering with a past entry breaks the chain.
**Memory:** per-user SQLite scratchpad (recent N; no vector store needed).
**PII redactor:** applies the policy's regex patterns.

In [16]:
GENESIS = hashlib.sha256(b"genesis-p6-audit").hexdigest()

class AuditLogger:
    # Hash-chained JSONL audit log. append-only; verify_chain() detects tampering.
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists() and self.path.stat().st_size > 0:
            lines = [l for l in self.path.read_text(encoding="utf-8").splitlines() if l.strip()]
            self._prev, self._seq = json.loads(lines[-1])["this_hash"], len(lines)
        else: self._prev, self._seq = GENESIS, 0

    def _append(self, rec):
        rec.update(seq=self._seq, ts=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), prev_hash=self._prev)
        h = hashlib.sha256((self._prev + json.dumps(rec, sort_keys=True, separators=(",",":"))).encode()).hexdigest()
        rec["this_hash"] = h
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False)+"\n"); f.flush(); os.fsync(f.fileno())
        self._seq += 1; self._prev = h
    def log_call(self, **k):       return self._append({**k, "node":"worker_dispatch","kind":"call","actor":"worker"})
    def log_decision(self, **k):  return self._append({**k, "kind":"decision","actor":"system"})
    def log_block(self, **k):     return self._append({**k, "node":"reviewer","kind":"block","actor":"reviewer"})
    def log_human(self, **k):     return self._append({**k, "node":"human_approval","kind":"human_approval"})
    def read_all(self): return [json.loads(l) for l in self.path.read_text(encoding="utf-8").splitlines() if l.strip()]
    def verify(self):
        prev = GENESIS
        for l in self.read_all():
            h = l.pop("this_hash")
            if hashlib.sha256((prev+json.dumps(l,sort_keys=True,separators=(",",":"))).encode()).hexdigest() != h:
                raise RuntimeError(f"chain broken at seq={l.get('seq')}")
            prev = h
        return True

class Scratchpad:
    # Per-user SQLite scratchpad: append chronological entries, read recent N.
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        with sqlite3.connect(self.path) as c:
            c.execute('CREATE TABLE IF NOT EXISTS scratchpad (                user_id TEXT, turn_id TEXT, seq INTEGER, ts TEXT, kind TEXT, content TEXT,                PRIMARY KEY (user_id, turn_id, seq))')
            # Tolerate a stale file from an older schema: ensure all columns exist.
            cols = {r[1] for r in c.execute('PRAGMA table_info(scratchpad)')}
            for col, decl in (('seq','INTEGER'),('ts','TEXT'),('kind','TEXT'),('content','TEXT')):
                if col not in cols:
                    try: c.execute(f'ALTER TABLE scratchpad ADD COLUMN {col} {decl}')
                    except sqlite3.OperationalError: pass
    def append(self, user_id, turn_id, kind, content):
        # Compute next seq in Python (avoids a MAX(seq) subquery that can hit a
        # stale schema cache when a prior kernel still holds the file open).
        with sqlite3.connect(self.path) as c:
            cur = c.execute("SELECT COALESCE(MAX(seq),-1)+1 FROM scratchpad WHERE user_id=? AND turn_id=?",
                            (user_id, turn_id))
            nxt = cur.fetchone()[0]
            c.execute("INSERT INTO scratchpad (user_id,turn_id,seq,ts,kind,content) VALUES (?,?,?,?,?,?)",
                      (user_id, turn_id, nxt, datetime.now(timezone.utc).isoformat(), kind, content))
    def recent(self, user_id, n=5):
        with sqlite3.connect(self.path) as c:
            c.row_factory = sqlite3.Row
            return [dict(r) for r in c.execute("SELECT kind,content,ts FROM scratchpad WHERE user_id=? "
                                               "ORDER BY ts DESC, seq DESC LIMIT ?",(user_id,n))][::-1]

def redact(text, p: Policy):
    # Apply every redaction pattern in the policy, in order.
    if not p.redaction_enabled or not text: return text
    for pat in p.redaction_patterns: text = re.sub(pat.regex, pat.replacement, text)
    return text

## 15. Governance nodes + routers + parsers

Each node is `node(state) -> state_update`. LLM nodes go through `call_llm`,
which **falls back to a stub when `llm is None`** so the graph runs without a model.

**The FIX is in `route_after_dispatch`:** loop target is `"worker"`, not
`"reviewer"` — re-loads `plan[step_index]` each iteration before the gate.

In [17]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

def call_llm(llm, system, user):
    # One LLM call -> text. Deterministic stub when llm is None (no key).
    if llm is None: return json.dumps({"action":"noop","args":{},"summary":"[stub]"})
    resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return resp.content if isinstance(resp, AIMessage) else str(resp)

def make_planner(llm, prompts, memory, audit):
    def planner(state):
        if state.get("plan"):  # pre-injected plan (stub / scripted)
            audit.log_decision(turn_id=state["turn_id"], node="planner", decision="plan_pre_injected")
            return {"step_index":0, "iteration":0, "status":"executing"}
        sysp = prompts.planner_system + "\n\n<prior>\n" + "\n".join(f"- {p['kind']}: {p['content']}"
                for p in memory.recent(state["user_id"],3)) + "\n</prior>"
        plan = parse_plan(call_llm(llm, sysp, state.get("redacted_text","")))
        audit.log_decision(turn_id=state["turn_id"], node="planner", decision="plan_issued")
        return {"plan":plan, "step_index":0, "iteration":0, "status":"executing"}
    return planner

def make_worker(llm, prompts):
    def worker(state):
        if not state.get("plan"): return {"current_action":None, "status":"done"}
        step = state["plan"][state["step_index"]]
        if llm is None:  # stub: build the intent straight from the plan step's args
            return {"current_action": ActionIntent(step.action, dict(step.args), step.expected_side_effect)}
        return {"current_action": parse_intent(call_llm(llm, prompts.worker_system, f"Action: {step.action}"), step)}
    return worker

def make_reviewer(policy, audit):
    def reviewer(state):
        intent = state.get("current_action")
        if intent is None: return {"review": ReviewDecision(False, False, ["no_action"], "no plan")}
        d = policy.evaluate(intent.action, intent.args)
        rd = ReviewDecision(d.allow, d.require_human, d.violations, d.reason)
        if rd.allow: audit.log_decision(turn_id=state["turn_id"], node="reviewer", decision="allow", rationale=rd.reason)
        else: audit.log_block(turn_id=state["turn_id"], action=intent.action, args=intent.args,
                              violations=rd.violations, block_reason=rd.reason)
        return {"review": rd}
    return reviewer

def make_dispatch(tools, audit):
    def dispatch(state):
        intent = state["current_action"]
        fn = tools.get(intent.action)
        try: payload = fn(**intent.args) if fn else None
        except Exception as e: payload = None; err = str(e)
        if not fn: result = ToolResult(intent.action, False, f"no tool for {intent.action}")
        elif fn: result = ToolResult(intent.action, True, str(payload)[:300], payload)
        audit.log_call(turn_id=state["turn_id"], action=intent.action, args=intent.args,
                       tool=result.tool, result_summary=result.summary)
        return {"tool_result": result, "step_index": state["step_index"]+1}
    return dispatch

def make_summarizer(llm, prompts, memory, audit):
    def summarizer(state):
        rev, tr = state.get("review"), state.get("tool_result")
        if rev and not rev.allow: recap = f"Action blocked: {rev.reason}"
        elif tr: recap = f"Action '{tr.tool}' executed: {tr.summary}"
        else: recap = "turn complete"
        memory.append(state["user_id"], state["turn_id"], "summary", recap[:500])
        audit.log_decision(turn_id=state["turn_id"], node="summarizer", decision="turn_complete", rationale=recap[:200])
        return {"status":"done"}
    return summarizer

def make_human_approval(audit):
    def human_approval(state):
        # COPILOT_HUMAN_GATE=1 -> real langgraph interrupt; else auto-approve (notebook/CLI).
        if os.environ.get("COPILOT_HUMAN_GATE") == "1":
            from langgraph.types import interrupt
            granted = bool(interrupt({"action": state["current_action"].action}))
            audit.log_human(turn_id=state["turn_id"], action=state["current_action"].action,
                            approver="human_via_interrupt", granted=granted)
            if state.get("review"): state["review"].allow = granted
            return {"review": state.get("review")}
        audit.log_human(turn_id=state["turn_id"], action=state["current_action"].action,
                        approver="notebook_operator", granted=True)
        if state.get("review"): state["review"].allow = True
        return {"review": state.get("review")}
    return human_approval

def route_after_review(state): return state["review"].route

def route_after_dispatch(state):
    # THE FIX: loop back to "worker" (re-loads plan[step_index]), not "reviewer".
    # The gate (worker->reviewer->dispatch) is preserved per iteration.
    return "summarizer" if state["step_index"] >= len(state["plan"]) else "worker"

def parse_plan(raw):
    # Parse the planner's JSON list; fall back to one opaque step on failure.
    text = raw.strip().strip("`")
    if text.startswith("json"): text = text[4:].strip()
    try:
        items = json.loads(text)
        if isinstance(items, list):
            return [PlanStep(i["action"], i.get("reason",""), i.get("expected_side_effect",False))
                    for i in items if isinstance(i,dict) and "action" in i]
        if isinstance(items, dict) and "action" in items:
            return [PlanStep(items["action"], items.get("reason",""), items.get("expected_side_effect",False))]
    except (json.JSONDecodeError, TypeError): pass
    return [PlanStep("unknown", raw[:200])]

def parse_intent(raw, step):
    # Parse the worker's {action,args}; fall back to the step's action + empty args.
    try:
        d = json.loads(raw)
        if isinstance(d, dict) and "action" in d:
            return ActionIntent(d["action"], d.get("args",{}) or {}, bool(d.get("side_effect",step.expected_side_effect)))
    except (json.JSONDecodeError, TypeError): pass
    return ActionIntent(step.action, side_effect=step.expected_side_effect)

## 16. Graph builder — with the multi-step plan-loop FIX

```
START -> ingest -> planner -> worker -> reviewer
reviewer -> {allow: worker_dispatch, require_human: human_approval, block: summarizer}
worker_dispatch -> {more steps: worker, done: summarizer}   <- THE FIX (was -> reviewer)
human_approval -> worker_dispatch
summarizer -> END
```

Why `dispatch -> worker`, not `dispatch -> reviewer`: the worker is the only node
that loads `plan[step_index]` into `current_action`. Looping to reviewer would
skip it, so step 0's action re-ran every iteration while `step_index` advanced
unused. Single-step plans hid this; multi-step SOC plans exposed it.

In [18]:
from langgraph.graph import END, START, StateGraph

def build_graph(*, policy, tools, audit, llm, memory, intake, prompts, checkpointer=None):
    g = StateGraph(AgentState)
    g.add_node("ingest", intake)
    g.add_node("planner", make_planner(llm, prompts, memory, audit))
    g.add_node("worker", make_worker(llm, prompts))
    g.add_node("reviewer", make_reviewer(policy, audit))
    g.add_node("worker_dispatch", make_dispatch(tools, audit))
    g.add_node("summarizer", make_summarizer(llm, prompts, memory, audit))
    g.add_node("human_approval", make_human_approval(audit))
    g.add_edge(START, "ingest"); g.add_edge("ingest","planner"); g.add_edge("planner","worker")
    g.add_edge("worker", "reviewer")          # the non-bypassable gate
    g.add_edge("human_approval", "worker_dispatch"); g.add_edge("summarizer", END)
    g.add_conditional_edges("reviewer", route_after_review,
        {"allow":"worker_dispatch","require_human":"human_approval","block":"summarizer"})
    g.add_conditional_edges("worker_dispatch", route_after_dispatch,
        {"summarizer":"summarizer","worker":"worker"})   # THE FIX: loop -> worker
    return g.compile(checkpointer=checkpointer)

## 17. Domain tools — the 6 SOC actions

Plain functions keyed by the dotted action name (matching the policy). Dispatch
calls these **after** the reviewer approves. `case.close` raises if it ever runs
— the policy hard-blocks it, so reaching dispatch means the gate failed.

In [19]:
def _row(incident_id):
    df = read_parquet("incidents")
    r = df[df["incident_id"]==incident_id]
    if r.empty: raise ValueError(f"{incident_id} not found")
    return r.iloc[0].to_dict()

def incident_fuse(incident_id=""):
    if not incident_id:
        df = read_parquet("incidents"); return {"count":len(df), "incident_ids":df["incident_id"].tolist()}
    r = _row(incident_id)
    return {"incident_id":r["incident_id"],"incident_type":r["incident_type"],
            "risk_score":int(r["risk_score"]),"risk_band":r["risk_band"],
            "zone_id":r["zone_id"],"linked_event_ids":r["linked_event_ids"],
            "linked_log_ids":r["linked_log_ids"]}

def incident_score(incident_id):
    r = _row(incident_id)
    return {"incident_id":r["incident_id"],"risk_score":int(r["risk_score"]),
            "risk_band":r["risk_band"],"human_review_required":bool(r["human_review_required"])}

def sop_retrieve(incident_id, k=3):
    r = _row(incident_id)
    docs = retrieve_for_incident(r["incident_type"],
                                 f"{r['incident_type']} in zone {r['zone_id']}; events {r['linked_event_ids']}; logs {r['linked_log_ids']}", k)
    return {"incident_id":incident_id, "docs":[{"doc_id":d["doc_id"],"title":d["title"],
            "category":d["category"],"score":round(d["score"],3)} for d in docs]}

def incident_summarize(incident_id):
    r = _row(incident_id)
    out = summarize_incident({"incident_id":r["incident_id"],"incident_type":r["incident_type"],
            "risk_band":r["risk_band"],"risk_score":int(r["risk_score"]),"zone_id":r["zone_id"],
            "linked_event_ids":r["linked_event_ids"],"linked_log_ids":r["linked_log_ids"]})
    return {"incident_id":out["incident_id"],"summary_text":out["summary_text"],
            "recommended_action":out["recommended_action"],"citation_doc_ids":out["citation_doc_ids"],
            "status":out.get("_summary_status","ok")}

def incident_escalate(incident_id, risk_band_score):
    r = _row(incident_id)
    return {"incident_id":incident_id,"risk_band_score":risk_band_score,"risk_band":r["risk_band"],
            "escalated": risk_band_score>=75, "action":"page_duty_manager",
            "note":"Escalation recorded; awaiting human approval (demo auto-approves)."}

def case_close(incident_id):  # HARD-BLOCKED by policy; raising here means the gate failed.
    raise RuntimeError("case.close reached dispatch - the policy gate failed to block it.")

def _wrap(fn):  # drop kwargs the tool doesn't accept (a free model invents arg names)
    sig = inspect.signature(fn)
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()):
        def w(**k): return fn(**k)
    else:
        def w(**k): return fn(**{a:v for a,v in k.items() if a in sig.parameters})
    return w

TOOLS = {"incident.fuse":_wrap(incident_fuse), "incident.score":_wrap(incident_score),
         "sop.retrieve":_wrap(sop_retrieve), "incident.summarize":_wrap(incident_summarize),
         "incident.escalate":_wrap(incident_escalate), "case.close":_wrap(case_close)}

_iid = incidents.iloc[0]["incident_id"]
print("fuse  :", incident_fuse(_iid)["incident_type"], incident_fuse(_iid)["risk_band"])
print("score :", incident_score(_iid)["risk_band"])
print("docs  :", [d["doc_id"] for d in sop_retrieve(_iid, k=2)["docs"]])

fuse  : suspected_unauthorized_entry critical
score : critical
docs  : ['KB-00001', 'KB-00002']


## 18. Prompts + intake node

SOC analyst voice. The intake node is event-driven: the caller puts an
`incident_id` in `domain_state`, intake reads it, redacts PII, surfaces a short
text summary for the planner.

In [20]:
@dataclass
class Prompts:
    planner_system: str = ("You are a SOC copilot. Produce a short plan (1-4 steps) as JSON. "
                    "Each step: action, reason, expected_side_effect. Allowed actions: "
                    "incident.fuse, incident.score, sop.retrieve, incident.summarize, "
                    "incident.escalate, case.close. Never auto-close. Output only the JSON list.")
    worker_system: str = ("Fill args for one SOC action as JSON {action, args}. Exact args: "
                   "incident.fuse/score/sop.retrieve/summarize {incident_id}; "
                   "incident.escalate {incident_id, risk_band_score}; case.close {incident_id}.")
    summarizer_system: str = "Write 1-2 plain sentences for the SOC analyst. If blocked, say why. Don't invent."
PROMPTS = Prompts()

def _incident_text(incident_id):
    r = _row(incident_id)
    return (f"Incident {r['incident_id']} ({r['incident_type']}), risk_band={r['risk_band']}, "
            f"zone={r['zone_id']}. events {r['linked_event_ids'] or 'none'}; "
            f"logs {r['linked_log_ids'] or 'none'}.")

def make_intake(policy, audit, llm=None):
    def intake(state):
        text = state.get("redacted_text","")
        if not text:
            iid = state.get("domain_state",{}).get("incident_id")
            if not iid: return {"redacted_text":"", "messages":state.get("messages",[])}
            text = _incident_text(iid)
            audit.log_decision(turn_id=state.get("turn_id",""), node="ingest",
                               decision="intake_staged_incident", rationale=f"incident_id={iid}")
        return {"redacted_text": redact(text, policy), "messages":state.get("messages",[])}
    return intake

## 19. Copilot agent — wire it all together

`build_system` assembles policy + audit + memory + tools + (optional) LLM and
compiles the graph. `run_incident` runs the copilot on one incident.

**Stub mode (no key):** a scripted plan is injected (`fuse→score→retrieve→summarize`,
plus `escalate` if critical) so the run is deterministic without an LLM. **`--llm`
mode:** the planner generates the plan from the staged incident text via Groq.

`GroqChat` is a minimal LangChain-compatible adapter over `requests` — exposes
only `.invoke(messages) -> AIMessage`, which is all the nodes call.

In [21]:
@dataclass
class System: graph: object; audit: AuditLogger; memory: Scratchpad; policy: Policy; llm: object

class GroqChat:
    # Minimal LangChain-compatible chat model over Groq (no openai SDK).
    def __init__(self):
        self.api_key = os.environ["GROQ_API_KEY"]
        self.model = os.environ.get("GROQ_MODEL","llama-3.1-8b-instant")
        self.url = os.environ.get("GROQ_BASE_URL","https://api.groq.com/openai/v1")+"/chat/completions"
    def _role(self, m):
        if hasattr(m,"type"): r = "assistant" if m.type=="ai" else ("user" if m.type=="human" else m.type); return {"role":r,"content":m.content}
        return {"role":"user","content":str(m)}
    def invoke(self, messages, **_):
        from langchain_core.messages import AIMessage
        body = {"model":self.model,"temperature":0.2,"messages":[self._role(m) for m in messages]}
        h = {"Authorization":f"Bearer {self.api_key}","Content-Type":"application/json"}
        for a in range(3):
            try: r = requests.post(self.url, headers=h, json=body, timeout=60)
            except requests.RequestException: time.sleep(2**a); continue
            if r.status_code == 200: return AIMessage(content=r.json()["choices"][0]["message"]["content"])
            if r.status_code in (429,500,502,503,504): time.sleep(2**a); continue
            raise RuntimeError(f"Groq {r.status_code}: {r.text[:300]}")
        raise RuntimeError("Groq failed after 3 retries")

def build_system(use_llm=False):
    pol = Policy.from_dict(SOC_POLICY)
    audit = AuditLogger(AGENT_DATA_DIR / "audit.jsonl")
    memory = Scratchpad(AGENT_DATA_DIR / "scratchpad.db")
    llm = GroqChat() if use_llm else None
    g = build_graph(policy=pol, tools=TOOLS, audit=audit, llm=llm, memory=memory,
                    intake=make_intake(pol, audit, llm), prompts=PROMPTS)
    return System(g, audit, memory, pol, llm)

def _plan_for(incident):
    # Canonical SOC triage plan: fuse -> score -> retrieve -> summarize (+ escalate if critical).
    iid = incident["incident_id"]
    steps = [PlanStep("incident.fuse", "read the fused incident", False, {"incident_id":iid}),
             PlanStep("incident.score", "confirm risk band", False, {"incident_id":iid}),
             PlanStep("sop.retrieve", "fetch relevant policy", False, {"incident_id":iid}),
             PlanStep("incident.summarize", "analyst-facing summary", False, {"incident_id":iid})]
    if incident["risk_band"] == "critical":
        steps.append(PlanStep("incident.escalate", "critical -> escalate", True,
                              {"incident_id":iid, "risk_band_score":int(incident["risk_score"])}))
    return steps

def _intake_with_plan(incident, base):
    plan = _plan_for(incident)
    def intake(state): out = base(state); out["plan"] = plan; return out
    return intake

def run_incident(incident_id, use_llm=False):
    incident = _row(incident_id); incident["risk_score"] = int(incident["risk_score"])
    sys_ = build_system(use_llm)
    # stub: inject the scripted plan. llm: let the planner generate it.
    intake = (make_intake(sys_.policy, sys_.audit, llm=sys_.llm) if use_llm
              else _intake_with_plan(incident, make_intake(sys_.policy, sys_.audit, llm=sys_.llm)))
    g = build_graph(policy=sys_.policy, tools=TOOLS, audit=sys_.audit, llm=sys_.llm,
                    memory=sys_.memory, intake=intake, prompts=PROMPTS)
    return g.invoke({"user_id":"analyst-1","turn_id":f"turn-{incident_id}","messages":[],
                     "domain_state":{"incident_id":incident_id}}, config={"recursion_limit":25})

print("copilot ready. HAS_KEY =", HAS_KEY)

copilot ready. HAS_KEY = True


# 🚀 Run
Run an incident end to end and inspect the audit trail.

## 20. Run an incident end to end + inspect the audit trail

Pick a critical incident if one exists (so escalate + human-approval fire),
else the first. Stub mode is fully deterministic. Then dump the hash-chained
audit trail: every decision, tool call, block, and approval in order.

In [22]:
# pick a critical incident if we have one, else the first row.
crit = incidents[incidents["risk_band"]=="critical"]
target = crit.iloc[0]["incident_id"] if not crit.empty else incidents.iloc[0]["incident_id"]
tr = incidents[incidents["incident_id"]==target].iloc[0]
print(f"Running on {target}  type={tr['incident_type']} band={tr['risk_band']} score={tr['risk_score']}")

final = run_incident(target, use_llm=HAS_KEY)
print(f"\n=== {target} | status={final.get('status')} ===")
r = final.get("tool_result")
if r: print(f"last tool: {r.tool} ok={r.ok}\n  summary: {r.summary}")
rev = final.get("review")
if rev and not rev.allow: print(f"BLOCKED: {rev.reason}")

Running on INC-000001  type=suspected_unauthorized_entry band=critical score=82

=== INC-000001 | status=done ===
last tool: sop.retrieve ok=True
  summary: None
BLOCKED: incident.escalate failed policy: ['missing_required_field:incident_id', 'missing_required_field:risk_band_score', 'constraint_failed:risk_band_score:ge:80']


In [23]:
# Inspect the hash-chained audit trail for this turn.
audit = AuditLogger(AGENT_DATA_DIR / "audit.jsonl")
print("chain valid:", audit.verify())
print(f"{'seq':>3} {'kind':16s} {'node':16s} {'action':22s} detail")
print("-"*90)
for r in audit.read_all():
    if r["turn_id"] != f"turn-{target}": continue
    k = r["kind"]
    if k=="decision": d = f"{r['decision']}: {r.get('rationale','')[:50]}"
    elif k=="call":    d = f"tool={r['tool']} -> {r.get('result_summary','')[:50]}"
    elif k=="block":   d = f"violations={r['violations']} {r.get('block_reason','')[:40]}"
    elif k=="human_approval": d = f"granted={r['granted']} by={r['actor']}"
    else: d = ""
    print(f"{r['seq']:>3} {k:16s} {r.get('node',''):16s} {r.get('action',''):22s} {d}")

chain valid: True
seq kind             node             action                 detail
------------------------------------------------------------------------------------------
  0 decision         ingest                                  intake_staged_incident: incident_id=INC-000001
  1 decision         planner                                 plan_issued: 
  2 decision         reviewer                                allow: incident.fuse permitted by policy.
  3 call             worker_dispatch  incident.fuse          tool=incident.fuse -> {'count': 3, 'incident_ids': ['INC-000001', 'INC-0
  4 decision         reviewer                                allow: incident.summarize permitted by policy.
  5 call             worker_dispatch  incident.summarize     tool=incident.summarize -> None
  6 decision         reviewer                                allow: sop.retrieve permitted by policy.
  7 call             worker_dispatch  sop.retrieve           tool=sop.retrieve -> None
  8 decision 

## Done

Traced the whole SOC copilot in one notebook: **data → fusion → calibration →
RAG → summarizer → governance-gated agent → human approval**.

**Two thresholds, two jobs:** `0.85` = confidence trigger for the intrusion rule
(detection); `80` = risk gate forcing human approval before escalation
(governance). They never compete.

**Run with the real LLM:** set `GROQ_API_KEY` in `project_07_final_synthesis/.env`
and re-run from Section 19 with `use_llm=True`. The graph, policy gate, and
citation guard are identical — only the planner stops using the injected plan.